# 일별 품목별 매출이익 TSV 추출

- 집계 컬럼: Box, 중량(Kg), 매출금액, 매입금액, 매출이익
- Box: 순 박스 수량 (출고 - 반품)
- 중량(Kg): 순 중량 (출고 - 반품)
- 매입금액: 중량(Kg) x 매입단가(pit_cost)
- 매출이익: 매출금액 - 매입금액

In [ ]:
import pyodbc
import pandas as pd
from pathlib import Path

CONN_STR = "DRIVER={SQL Server};SERVER=211.47.183.140,3341;DATABASE=SMVATPDA;UID=sa;PWD=0m2a2c0"
START_DATE = "20200101"
END_DATE = "20251231"
OUTPUT_TSV = Path("C:/Users/OWNER/Desktop/archive/06_project/06_SM프로그램자동화/01_쿼리예시/일별_품목별_매출이익.tsv")

In [ ]:
query = """
SELECT
    daily.tdate AS [일자],
    daily.icode AS [품목코드],
    i.ite_name AS [품목명],
    i.ite_sanzi + '(' + i.ite_dung + ')' AS [산지],
    SUM(daily.cBox) AS [Box],
    SUM(daily.cQty) AS [중량(Kg)],
    SUM(daily.cAmt) AS [매출금액],
    SUM(daily.cQty * ISNULL(p.pit_cost, 0)) AS [매입금액],
    SUM(daily.cAmt - (daily.cQty * ISNULL(p.pit_cost, 0))) AS [매출이익]
FROM (
    SELECT
        smp.icode,
        smp.tdate,
        smp.tyyyy,
        smp.tmm,
        SUM(ISNULL(smp.chu_box, 0) - ISNULL(smp.cba_box, 0)) AS cBox,
        SUM(ISNULL(smp.chu_qty, 0) - ISNULL(smp.cba_qty, 0)) AS cQty,
        SUM(ISNULL(smp.chu_amt, 0) - ISNULL(smp.cba_amt, 0)) AS cAmt
    FROM (
        SELECT
            SUBSTRING(chu_yymmdd, 1, 8) AS tdate,
            SUBSTRING(chu_yymmdd, 1, 4) AS tyyyy,
            SUBSTRING(chu_yymmdd, 5, 2) AS tmm,
            chu_icode AS icode,
            chu_box,
            chu_qty,
            chu_won AS chu_amt,
            0 AS cba_box,
            0 AS cba_qty,
            0 AS cba_amt
        FROM chul
        WHERE chu_flag = '1'
          AND chu_icode BETWEEN '' AND 'zzzzzzzz'
          AND chu_zcode LIKE '%'
          AND chu_yymmdd BETWEEN ? AND ?

        UNION ALL

        SELECT
            SUBSTRING(cba_yymmdd, 1, 8) AS tdate,
            SUBSTRING(cba_yymmdd, 1, 4) AS tyyyy,
            SUBSTRING(cba_yymmdd, 5, 2) AS tmm,
            cba_icode AS icode,
            0 AS chu_box,
            0 AS chu_qty,
            0 AS chu_amt,
            cba_box,
            cba_qty,
            cba_won AS cba_amt
        FROM cban
        WHERE cba_icode BETWEEN '' AND 'zzzzzzzz'
          AND cba_zcode LIKE '%'
          AND cba_yymmdd BETWEEN ? AND ?
    ) smp
    GROUP BY smp.icode, smp.tdate, smp.tyyyy, smp.tmm
) daily
LEFT JOIN pitem p
       ON p.pit_icode = daily.icode
      AND p.pit_yyyy = daily.tyyyy
      AND p.pit_mm = daily.tmm
LEFT JOIN item i
       ON i.ite_code = daily.icode
WHERE SUBSTRING(daily.icode, 1, 1) BETWEEN '' AND 'Z'
  AND SUBSTRING(daily.icode, 2, 1) BETWEEN '' AND 'Z'
  AND SUBSTRING(daily.icode, 2, 2) BETWEEN '' AND 'ZZ'
  AND SUBSTRING(daily.icode, 2, 1) + SUBSTRING(daily.icode, 4, 2) BETWEEN '' AND 'ZZZ'
  AND SUBSTRING(daily.icode, 6, 3) BETWEEN '' AND 'ZZZ'
GROUP BY
    daily.tdate,
    daily.icode,
    i.ite_name,
    i.ite_sanzi + '(' + i.ite_dung + ')'
ORDER BY
    daily.tdate,
    i.ite_name,
    i.ite_sanzi + '(' + i.ite_dung + ')';
"""

In [ ]:
with pyodbc.connect(CONN_STR, timeout=10) as conn:
    df = pd.read_sql(query, conn, params=[START_DATE, END_DATE, START_DATE, END_DATE])

if not df.empty:
    df["일자"] = pd.to_datetime(df["일자"], format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")

OUTPUT_TSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_TSV, sep="\t", index=False, encoding="utf-8-sig")

print(f"rows: {len(df):,}")
print(f"saved: {OUTPUT_TSV}")
df.head()